In [1]:
!apt-get update -y
!apt-get install -y tesseract-ocr poppler-utils

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,806 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/u

In [2]:
!pip install pdf2image pytesseract Pillow opencv-python-headless numpy pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 20.7 MB/s eta 0:00:00


In [3]:
import os
print(os.getcwd())

/content


In [4]:
from pathlib import Path
import shutil

BASE_DIR = Path("/content")

#making output and input dir
INPUT_DIR = BASE_DIR / "input"
UPLOAD_DIR = BASE_DIR / "output"

INPUT_DIR.mkdir(exist_ok=True)
UPLOAD_DIR.mkdir(exist_ok=True)

pdf_files = list(INPUT_DIR.glob("*.pdf"))
#copying from input folder to output folder
if not pdf_files:
    print("No PDF file found in the input folder.")
else:
    for pdf_path in pdf_files:
        stored_path = UPLOAD_DIR / pdf_path.name
        shutil.copy2(pdf_path, stored_path)

        print(f"Stored: {pdf_path.name} -> {stored_path}")

Stored: SI_CHRONICLES_25.pdf -> /content/output/SI_CHRONICLES_25.pdf


In [5]:
from pdf2image import convert_from_path, pdfinfo_from_path

if not pdf_files:
    print("No PDF file found in uploads folder.")
else:
    for pdf_path in pdf_files:
        output_image_dir = BASE_DIR / f"{pdf_path.stem}_images"
        output_image_dir.mkdir(exist_ok=True)

        total_pages = pdfinfo_from_path(pdf_path)["Pages"]

        for page_number in range(1, total_pages + 1):
          page_image = convert_from_path(pdf_path, dpi=300, first_page=page_number, last_page=page_number)[0]
          image_path = output_image_dir / f"page_{page_number}.png"
          page_image.save(image_path, "PNG")

          page_image.close()#deleting the image to restore memory
          del page_image
          gc.collect()
          print(f"Saved page {page_number}/{total_pages} -> {image_path}")

        print(f"Converted {pdf_path.name} into images at: {output_image_dir}")

Saved page 1/179 -> /content/SI_CHRONICLES_25_images/page_1.png
Saved page 2/179 -> /content/SI_CHRONICLES_25_images/page_2.png
Saved page 3/179 -> /content/SI_CHRONICLES_25_images/page_3.png
Saved page 4/179 -> /content/SI_CHRONICLES_25_images/page_4.png
Saved page 5/179 -> /content/SI_CHRONICLES_25_images/page_5.png
Saved page 6/179 -> /content/SI_CHRONICLES_25_images/page_6.png
Saved page 7/179 -> /content/SI_CHRONICLES_25_images/page_7.png
Saved page 8/179 -> /content/SI_CHRONICLES_25_images/page_8.png
Saved page 9/179 -> /content/SI_CHRONICLES_25_images/page_9.png
Saved page 10/179 -> /content/SI_CHRONICLES_25_images/page_10.png
Saved page 11/179 -> /content/SI_CHRONICLES_25_images/page_11.png
Saved page 12/179 -> /content/SI_CHRONICLES_25_images/page_12.png
Saved page 13/179 -> /content/SI_CHRONICLES_25_images/page_13.png
Saved page 14/179 -> /content/SI_CHRONICLES_25_images/page_14.png
Saved page 15/179 -> /content/SI_CHRONICLES_25_images/page_15.png
Saved page 16/179 -> /conten

In [7]:
from pathlib import Path
from PIL import Image, ImageFilter, ImageOps
import cv2
import numpy as np

image_folder = output_image_dir

if not image_folder:
    print("No image folders found.")
else:
      cleaned_folder = BASE_DIR / f"{image_folder.name}_cleaned"
      cleaned_folder.mkdir(exist_ok=True)
      image_files = list(image_folder.glob("*.png"))
      for image_path in image_files:
          image = Image.open(image_path)
          # Convert image to grayscale
          image = ImageOps.grayscale(image)
          # Increase contrast
          image = ImageOps.autocontrast(image)
          # Light noise reduction
          image = image.filter(ImageFilter.MedianFilter(size=3))
          # Convert Pillow image to OpenCV format
          image_array = np.array(image)
          # Convert to black and white using adaptive threshold
          cleaned_array = cv2.adaptiveThreshold(
              image_array,
              255,
              cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
              cv2.THRESH_BINARY,
              31,
              15
          )
          cleaned_image = Image.fromarray(cleaned_array)
          cleaned_path = cleaned_folder / image_path.name
          cleaned_image.save(cleaned_path)
          print(f"Cleaned: {image_path.name} -> {cleaned_path}")

          image.close()
          del image_array, cleaned_array
          gc.collect()

      print("No image folders found.")

Cleaned: page_125.png -> /content/SI_CHRONICLES_25_images_cleaned/page_125.png
Cleaned: page_17.png -> /content/SI_CHRONICLES_25_images_cleaned/page_17.png
Cleaned: page_131.png -> /content/SI_CHRONICLES_25_images_cleaned/page_131.png
Cleaned: page_165.png -> /content/SI_CHRONICLES_25_images_cleaned/page_165.png
Cleaned: page_51.png -> /content/SI_CHRONICLES_25_images_cleaned/page_51.png
Cleaned: page_176.png -> /content/SI_CHRONICLES_25_images_cleaned/page_176.png
Cleaned: page_143.png -> /content/SI_CHRONICLES_25_images_cleaned/page_143.png
Cleaned: page_59.png -> /content/SI_CHRONICLES_25_images_cleaned/page_59.png
Cleaned: page_110.png -> /content/SI_CHRONICLES_25_images_cleaned/page_110.png
Cleaned: page_58.png -> /content/SI_CHRONICLES_25_images_cleaned/page_58.png
Cleaned: page_52.png -> /content/SI_CHRONICLES_25_images_cleaned/page_52.png
Cleaned: page_91.png -> /content/SI_CHRONICLES_25_images_cleaned/page_91.png
Cleaned: page_123.png -> /content/SI_CHRONICLES_25_images_cleane

In [9]:
import re

cleaned_image_files = sorted(
    cleaned_folder.glob("*.png"),
    key=lambda p: int(re.search(r"\d+", p.stem).group())
)

# sanity check - print to confirm order before you trust it
for f in cleaned_image_files:
    print(f.name)

page_1.png
page_2.png
page_3.png
page_4.png
page_5.png
page_6.png
page_7.png
page_8.png
page_9.png
page_10.png
page_11.png
page_12.png
page_13.png
page_14.png
page_15.png
page_16.png
page_17.png
page_18.png
page_19.png
page_20.png
page_21.png
page_22.png
page_23.png
page_24.png
page_25.png
page_26.png
page_27.png
page_28.png
page_29.png
page_30.png
page_31.png
page_32.png
page_33.png
page_34.png
page_35.png
page_36.png
page_37.png
page_38.png
page_39.png
page_40.png
page_41.png
page_42.png
page_43.png
page_44.png
page_45.png
page_46.png
page_47.png
page_48.png
page_49.png
page_50.png
page_51.png
page_52.png
page_53.png
page_54.png
page_55.png
page_56.png
page_57.png
page_58.png
page_59.png
page_60.png
page_61.png
page_62.png
page_63.png
page_64.png
page_65.png
page_66.png
page_67.png
page_68.png
page_69.png
page_70.png
page_71.png
page_72.png
page_73.png
page_74.png
page_75.png
page_76.png
page_77.png
page_78.png
page_79.png
page_80.png
page_81.png
page_82.png
page_83.png
page_84.png
p

In [10]:
import pytesseract
from PIL import Image

page_pdf_bytes = []  # holds each page's single-page PDF, in order

for image_path in cleaned_image_files:  # from the sorted list you already built
    image = Image.open(image_path)

    pdf_bytes = pytesseract.image_to_pdf_or_hocr(image, extension="pdf")
    page_pdf_bytes.append(pdf_bytes)

    image.close()
    print(f"OCR'd: {image_path.name}")

print(f"Done - {len(page_pdf_bytes)} page(s) OCR'd, ready to merge")

OCR'd: page_1.png
OCR'd: page_2.png
OCR'd: page_3.png
OCR'd: page_4.png
OCR'd: page_5.png
OCR'd: page_6.png
OCR'd: page_7.png
OCR'd: page_8.png
OCR'd: page_9.png
OCR'd: page_10.png
OCR'd: page_11.png
OCR'd: page_12.png
OCR'd: page_13.png
OCR'd: page_14.png
OCR'd: page_15.png
OCR'd: page_16.png
OCR'd: page_17.png
OCR'd: page_18.png
OCR'd: page_19.png
OCR'd: page_20.png
OCR'd: page_21.png
OCR'd: page_22.png
OCR'd: page_23.png
OCR'd: page_24.png
OCR'd: page_25.png
OCR'd: page_26.png
OCR'd: page_27.png
OCR'd: page_28.png
OCR'd: page_29.png
OCR'd: page_30.png
OCR'd: page_31.png
OCR'd: page_32.png
OCR'd: page_33.png
OCR'd: page_34.png
OCR'd: page_35.png
OCR'd: page_36.png
OCR'd: page_37.png
OCR'd: page_38.png
OCR'd: page_39.png
OCR'd: page_40.png
OCR'd: page_41.png
OCR'd: page_42.png
OCR'd: page_43.png
OCR'd: page_44.png
OCR'd: page_45.png
OCR'd: page_46.png
OCR'd: page_47.png
OCR'd: page_48.png
OCR'd: page_49.png
OCR'd: page_50.png
OCR'd: page_51.png
OCR'd: page_52.png
OCR'd: page_53.png
OC

In [11]:
from pathlib import Path
from pypdf import PdfWriter
import io

OUTPUT_DIR = Path("searchable_pdfs")
OUTPUT_DIR.mkdir(exist_ok=True)

output_pdf_path = OUTPUT_DIR / "final_searchable.pdf"

writer = PdfWriter()

for pdf_bytes in page_pdf_bytes:
    pdf_stream = io.BytesIO(pdf_bytes)
    writer.append(pdf_stream)

with open(output_pdf_path, "wb") as output_pdf:
    writer.write(output_pdf)

print(f"Searchable PDF created: {output_pdf_path}")

Searchable PDF created: searchable_pdfs/final_searchable.pdf


In [13]:
from google.colab import files
files.download("searchable_pdfs/final_searchable.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>